# Single-waveform original VMD then dynamic STVMD

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import re
import warnings
import matplotlib.pyplot as plt
import numpy as np
import scipy
from IPython.display import display
from numba import jit, prange
from scipy.fft import irfft, rfft
from tqdm import tqdm

In [ ]:
INPUT_FILE = Path("5m.TXT")
DIRECTION = "Tran"

K = 4
ALPHA = 50.0
TAU = 1e-5
TOL = 1e-9
MAX_ITERS = 1000

VMD_N_FFT = 64
STVMD_WINDOW_LENGTH = 512

PLOT_MAX_HZ = 200.0
SAVE_OUTPUTS = True

In [ ]:
@dataclass(frozen=True)
class SingleWaveform:
    path: Path
    metadata: dict
    fs: float
    pretrigger_seconds: float
    direction: str
    time_s: np.ndarray
    values: np.ndarray


def _header_metadata(lines):
    metadata = {}
    for line in lines:
        normalized = line.strip().strip("\"'")
        if ":" not in normalized:
            continue
        key, value = normalized.split(":", 1)
        metadata[key.strip()] = value.strip()
    return metadata


def _numeric_rows_after_header(lines):
    header_index = None
    required = {"Tran", "Vert", "Long"}
    for index, line in enumerate(lines):
        words = set(line.strip().strip("\"'").split())
        if required.issubset(words):
            header_index = index
            break
    if header_index is None:
        raise ValueError("Could not find Tran Vert Long data header")

    rows = []
    for line in lines[header_index + 1:]:
        fields = line.strip().strip("\"'").split()
        if len(fields) < 3:
            continue
        try:
            rows.append([float(value) for value in fields[:3]])
        except ValueError:
            continue
    if not rows:
        raise ValueError("No numeric waveform rows found after data header")
    return np.asarray(rows, dtype=float)


def load_single_waveform(path, direction):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    if direction not in {"Tran", "Vert", "Long"}:
        raise ValueError("direction must be one of Tran, Vert, or Long")

    lines = path.read_text(
        encoding="utf-8-sig", errors="replace"
    ).splitlines()
    metadata = _header_metadata(lines)
    number_pattern = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"
    sample_rate_match = re.search(
        number_pattern, metadata.get("Sample Rate", "")
    )
    if sample_rate_match is None:
        raise ValueError("Sample Rate metadata is missing or invalid")
    fs = float(sample_rate_match.group())
    if not np.isfinite(fs) or fs <= 0:
        raise ValueError("Sample Rate must be a finite positive number")

    pretrigger_match = re.search(
        number_pattern, metadata.get("Pre-trigger Length", "")
    )
    pretrigger_seconds = (
        abs(float(pretrigger_match.group()))
        if pretrigger_match is not None
        else 0.0
    )

    rows = _numeric_rows_after_header(lines)
    column = {"Tran": 0, "Vert": 1, "Long": 2}[direction]
    values = rows[:, column]
    if not np.isfinite(values).all():
        raise ValueError("Waveform values must all be finite")
    time_s = np.arange(values.size) / fs - pretrigger_seconds
    return SingleWaveform(
        path=path,
        metadata=metadata,
        fs=fs,
        pretrigger_seconds=pretrigger_seconds,
        direction=direction,
        time_s=time_s,
        values=values,
    )

In [ ]:
@jit(nopython=True, cache=True)
def _buffer(x, out, nfft, hop_len):
    for i in prange(0, x.shape[-1]-nfft+1, hop_len):
        out[:,:,i]=x[:,i:i+nfft]
def buffer(x, nfft, hop_len, modulated=True):
    # x: channel, time
    if np.ndim(x)==1:
        x = x.reshape(1,-1)
    out = np.zeros((x.shape[0], nfft, x.shape[-1]-nfft+1))
    _buffer(x, out, nfft, hop_len)
    if modulated:
        out = np.roll(out, int(np.ceil(nfft / 2)), 1)
    return out

@jit(nopython=True, cache=True)
def _unbuffer(x, xbuf, window, hop_len):
    for i in prange(xbuf.shape[2]):
        n = i * hop_len
        x[:, n:n + len(window)] += xbuf[:, :, i] * window
        
def unbuffer(xbuf, window, hop_len, win_exp=1):
    if np.ndim(xbuf)==2:
        xbuf=np.expand_dims(xbuf,0)
    if win_exp == 0:
        window = 1
    elif win_exp != 1:
        window = window ** win_exp
    x = np.zeros((xbuf.shape[0], xbuf.shape[1] + xbuf.shape[2] - 1), dtype=xbuf.dtype)

    _unbuffer(x, xbuf, window, hop_len)
    return x

@jit(nopython=True, cache=True)
def _window_norm(wn, window, hop_len, n_fft, win_exp=1):
    max_hops = (len(wn) - n_fft) // hop_len + 1
    wpow = window ** (win_exp + 1)
    for i in range(max_hops):
        n = i * hop_len
        wn[n:n + n_fft] += wpow

def window_norm(window, hop_len, n_fft, N, win_exp=1):
    wn = np.zeros(N + n_fft - 1)
    _window_norm(wn, window, hop_len, n_fft, win_exp)
    return wn

In [ ]:
class VMD(object):
    def __init__(self, num_channel, n_fft= 128, alpha= 10, K= 3, init=1, tol=1e-05, tau= 0.1, maxiters=1000) -> None:
        self.alpha = alpha*np.ones(K)
        self.K=K
        self.n_fft = n_fft
        self.init=init
        self.tol=tol
        self.tau=tau
        self.maxiters = maxiters
        self.padwidth = ((n_fft-1)//2, (n_fft-1)//2) if (n_fft-1)%2==0 else ((n_fft-1)//2+1, (n_fft-1)//2)
        self.num_channel = num_channel
        self.u_hat_plus = None
        self.sum_uk = None
    
    def prepare_offline(self, x):
        assert np.ndim(x)<=2
        if np.ndim(x)==1:
            x = x.reshape(1, -1)
        self.len_x = x.shape[1]
        xp = np.pad(x, ([0, 0], self.padwidth), mode='reflect')
        f_hat_plus = rfft(xp, axis=1, workers=-1)
        return f_hat_plus
    
    def apply(self, f_hat_plus, omega=None):
        C, len_freqs = f_hat_plus.shape
        freqs = np.arange(1, len_freqs+1)/(len_freqs)
        omega_plus = np.zeros([self.maxiters, self.K])
        if omega is None:
            for i in range(self.K):
                omega_plus[0,i] = (1/self.K)*i
        else:
            omega_plus[0] = omega.ravel()
 
        u_hat_plus = np.zeros([self.maxiters, C, len_freqs, self.K],dtype=complex)
        sum_uk = np.zeros([C, len_freqs], dtype=complex)
        lambda_hat = np.zeros([self.maxiters, C, len_freqs], dtype = complex)
        
        pbar = tqdm(np.arange(self.maxiters-1))
        for iter in pbar:
            # n = np.mod(iter, 2)
            
            k = 0
            for c in np.arange(C):
                sum_uk[c,:] = u_hat_plus[np.mod(iter, 2), c,:,self.K-1] + sum_uk[c,:] - u_hat_plus[np.mod(iter, 2), c,:,0]
                u_hat_plus[np.mod(iter+1, 2), c,:,k] = (f_hat_plus[c, :] - sum_uk[c,:] - lambda_hat[np.mod(iter, 2), c,:]/2)/(1.+self.alpha[k]*(freqs - omega_plus[np.mod(iter, 2),k])**2)
            for k in np.arange(1,self.K):
                for c in np.arange(C):
                    sum_uk[c,:] = u_hat_plus[np.mod(iter+1, 2),c,:,k-1] + sum_uk[c,:] - u_hat_plus[np.mod(iter, 2),c,:,k]
                    u_hat_plus[np.mod(iter+1, 2),c,:,k] = (f_hat_plus[c,:] - sum_uk[c,:] - lambda_hat[np.mod(iter, 2),c,:]/2)/(1+self.alpha[k]*(freqs - omega_plus[np.mod(iter, 2),k])**2)
                # center frequencies
                if omega is not None:
                    omega_plus[np.mod(iter+1, 2),k] = omega_plus[0,k]
                else:
                    omega_plus[np.mod(iter+1, 2),k] = np.sum(np.einsum('j,ij', freqs,(abs(u_hat_plus[np.mod(iter+1, 2), :, :, k])**2)))/np.sum(abs(u_hat_plus[np.mod(iter+1, 2),:,:,k])**2)
            lambda_hat[np.mod(iter+1, 2),:,:] = lambda_hat[np.mod(iter, 2),:,:] + self.tau*(np.sum(u_hat_plus[np.mod(iter+1, 2),:,:,:],axis = 2)-f_hat_plus)

            uDiff = np.spacing(np.ones(self.K))
            for i in range(self.K):
                delta = u_hat_plus[0,:,:,i]-u_hat_plus[1,:,:,i]
                delta = delta.reshape(-1)
                uDiff[i] = (1/len(delta))*np.dot(delta,np.conj(delta)).real
            # for i in range(self.K):
            #     uDiff = uDiff + np.power(u_hat_plus[np.mod(iter, 2)]-u_hat_plus[np.mod(iter-1, 2)], 2).sum()/(len_freqs*self.K*C)
            uDiff = uDiff.mean(-1)
            if uDiff < self.tol and iter>2:
                break
            pbar.set_description("Processing "  + "{:10.9f}".format(uDiff) + ":")
        u_hat = u_hat_plus[np.mod(iter, 2)]
        omega = omega_plus[np.mod(iter, 2)]
        seqw = np.argsort(omega)
        u_hat = u_hat[:,:,seqw]
        omega = omega[seqw]
        return u_hat, omega
    
    def postprocess(self, u_hat):
        u = irfft(u_hat, n=self.len_x+np.sum(self.padwidth), axis=1, workers=-1).real
        u = np.transpose(u, (2, 0, 1))
        u = u[:, :, self.padwidth[0]:-self.padwidth[1]]
        return u

In [ ]:
class STVMD(object):
    def __init__(self, num_channel, n_fft= 128, cache_size=None, window_func=None, alpha= 10, K= 3, init=1, tol=1e-05, tau= 0.1, win_exp=1, maxiters=1000) -> None:
        # n_fft: odd integers
        self.hop_len = 1
        self.n_fft = n_fft
        self.win_len = n_fft
        self.alpha = alpha*np.ones(K)
        self.freqs = np.arange(1, n_fft//2+2)/(n_fft//2+1)
        self.K=K
        self.win_exp = win_exp
        self.init=init
        self.tol=tol
        self.tau=tau
        self.padwidth = ((n_fft-1)//2, (n_fft-1)//2) if (n_fft-1)%2==0 else ((n_fft-1)//2+1, (n_fft-1)//2)
        if window_func is None:
            self.window = scipy.signal.windows.dpss(self.win_len, max(4, self.win_len//8), sym=False)
        else:
            self.window = window_func
        self.maxiters = maxiters

        self.num_channel = num_channel
        
        if cache_size is None:
            self.cache_size = self.n_fft
        else:
            self.cache_size = cache_size
        self.buffer_cache = np.zeros((self.num_channel, self.cache_size))# channel, F, T
        self.u_hat_cache = np.zeros((self.num_channel, len(self.freqs), self.K, self.cache_size)) # C, len(self.freqs), self.K, N
        self.u_hat_plus = None
        self.sum_uk = None
    
    def prepare_online(self, x):
        assert np.ndim(x)<=2
        if np.ndim(x)==1:
            x = x.reshape(-1, 1)
        print(x.shape)
        print(self.buffer_cache.shape)
        new_dims = x.shape[-1]
        self.buffer_cache = np.roll(self.buffer_cache, -new_dims, axis=1)
        self.buffer_cache[:,-new_dims:]=x
        xp = np.pad(self.buffer_cache, ([0, 0], [0, (self.n_fft-1)//2]), mode='reflect')
        print(xp.shape)
        Sx = buffer(xp, self.n_fft, self.hop_len, modulated=False)
        # Sx *= ifftshift(self.window, axes=0).reshape((1, -1, 1))
        Sx *= self.window.reshape((1, -1, 1))
        print(Sx.shape)
        f_hat_plus = rfft(Sx, axis=1, workers=-1)
        return f_hat_plus, Sx
    
    def prepare_offline(self, x):
        assert np.ndim(x)<=2
        if np.ndim(x)==1:
            x = x.reshape(1, -1)
        # assert x.shape[-1]>=self.n_fft
        xp = np.pad(x, ([0, 0], self.padwidth), mode='reflect')
        self.buffer_cache = xp[:, -self.n_fft//2+1-self.n_fft:-self.n_fft//2+1]
        Sx = buffer(xp, self.n_fft, self.hop_len, modulated=False)
        # Sx *= ifftshift(self.window, axes=0).reshape((1, -1, 1))
        Sx *= self.window.reshape((1, -1, 1))
        f_hat_plus = rfft(Sx, axis=1, workers=-1)
        return f_hat_plus, Sx
    
    def apply(self, f_hat_plus, omega=None, dynamic=False):
        if dynamic:
            u_hat, omega = self.apply_dynamic(f_hat_plus)
        else:
            u_hat, omega = self.apply_nodynamic(f_hat_plus, omega)
        return u_hat, omega
    
    def apply_nodynamic(self, f_hat_plus, omega=None, cached=False):
        C, _, N = f_hat_plus.shape
        omega_plus = np.zeros([self.maxiters, self.K])
        if omega is None:
            for i in range(self.K):
                omega_plus[0,i] = (1/self.K)*i
        else:
            omega_plus[0,:] = omega.reshape(-1,1)
        
        if cached:
            u_hat_plus = np.zeros([2, C, len(self.freqs), self.K, N],dtype=complex)
            u_hat_plus[0,:,:,:,:-1]=self.u_hat_cache[:,:,:,1:]
        else:
            u_hat_plus = np.zeros([2, C, len(self.freqs), self.K, N],dtype=complex)
        sum_uk = np.zeros([N, C, len(self.freqs)], dtype=complex)
        lambda_hat = np.zeros([2, C, len(self.freqs), N], dtype = complex)
        
        pbar = tqdm(np.arange(self.maxiters-1))
        for iter in pbar:
            # n = np.mod(iter, 2)
            
            k = 0
            for c in np.arange(C):
                for sw in np.arange(N):
                    sum_uk[sw,c,:] = u_hat_plus[np.mod(iter, 2), c,:,self.K-1,sw] + sum_uk[sw, c,:] - u_hat_plus[np.mod(iter, 2), c,:,0, sw]
                    u_hat_plus[np.mod(iter+1, 2), c,:,k, sw] = (f_hat_plus[c, :, sw] - sum_uk[sw, c,:] - lambda_hat[np.mod(iter, 2), c,:,sw]/2)/(1.+self.alpha[k]*(self.freqs - omega_plus[np.mod(iter, 2),k])**2)
            for k in np.arange(1,self.K):
                for sw in np.arange(N):
                    for c in np.arange(C):
                        #accumulator
                        sum_uk[sw,c,:] = u_hat_plus[np.mod(iter+1, 2),c,:,k-1,sw] + sum_uk[sw,c,:] - u_hat_plus[np.mod(iter, 2),c,:,k,sw]
                        # mode spectrum
                        u_hat_plus[np.mod(iter+1, 2),c,:,k,sw] = (f_hat_plus[c,:,sw] - sum_uk[sw,c,:] - lambda_hat[np.mod(iter, 2),c,:,sw]/2)/(1+self.alpha[k]*(self.freqs - omega_plus[np.mod(iter, 2),k])**2)
                # center frequencies
                if omega is not None:
                    omega_plus[np.mod(iter+1, 2),k] = omega_plus[0,k]
                else:
                    omega_plus[np.mod(iter+1, 2),k] = np.sum(np.einsum('j,ijk', self.freqs,(abs(u_hat_plus[np.mod(iter+1, 2), :, :, k, :])**2)))/np.sum(abs(u_hat_plus[np.mod(iter+1, 2),:,:,k,:])**2)
            lambda_hat[np.mod(iter+1, 2),:,:,:] = lambda_hat[np.mod(iter, 2),:,:,:] + self.tau*(np.sum(u_hat_plus[np.mod(iter+1, 2),:,:,:,:],axis = 2)-f_hat_plus)
            
            uDiff = np.spacing(np.ones((C, N, self.K)))
            for i in range(self.K):
                for c in range(C):
                    for n in range(N):
                        delta = u_hat_plus[0,c,:,i,n]-u_hat_plus[1,c,:,i, n]
                        delta = delta.reshape(-1)
                        uDiff[c,n,i] = (1/len(delta))*np.dot(delta,np.conj(delta)).real
            uDiff = uDiff.mean(-1)
            uDiff = uDiff.max()
            if uDiff < self.tol and iter>2:
                break
            pbar.set_description("Processing "  + "{:10.9f}".format(uDiff) + ":")
        u_hat = u_hat_plus[np.mod(iter, 2)]
        omega = omega_plus[np.mod(iter, 2)]
        self.u_hat_cache = u_hat_plus[np.mod(iter, 2), :, :, :, -self.cache_size:]
        seqw = np.argsort(omega)
        u_hat = u_hat[:,:,seqw,:]
        omega = omega[seqw]
        return u_hat, omega
    
    def apply_dynamic(self, f_hat_plus, cached=False):
        C, F, N = f_hat_plus.shape
        freqs = np.arange(1, F+1)/F# self.freqs
        omega_plus = np.zeros([self.maxiters, self.K, N])
        for i in range(self.K):
            omega_plus[0,i,:] = (1/self.K)*i
            
        if cached:
            u_hat_plus = np.zeros([2, C, len(freqs), self.K, N],dtype=complex)
            u_hat_plus[0,:,:,:,:-1]=self.u_hat_cache[:,:,:,1:]
        else:
            u_hat_plus = np.zeros([2, C, len(freqs), self.K, N],dtype=complex)
        sum_uk = np.zeros([N, C, len(freqs)], dtype=complex)
        lambda_hat = np.zeros([2, C, len(freqs), N], dtype = complex)
        
        pbar = tqdm(np.arange(self.maxiters-1))
        for iter in pbar:
            # n = np.mod(iter, 2)
            
            k = 0
            for c in np.arange(C):
                for sw in np.arange(N):
                    sum_uk[sw,c,:] = u_hat_plus[np.mod(iter, 2), c,:,self.K-1,sw] + sum_uk[sw, c,:] - u_hat_plus[np.mod(iter, 2), c,:,0, sw]
                    u_hat_plus[np.mod(iter+1, 2), c,:,k, sw] = (f_hat_plus[c, :, sw] - sum_uk[sw, c,:] - lambda_hat[np.mod(iter, 2), c,:,sw]/2)/(1.+self.alpha[k]*(freqs - omega_plus[np.mod(iter, 2),k, sw])**2)
            for k in np.arange(1,self.K):
                for sw in np.arange(N):
                    for c in np.arange(C):
                        #accumulator
                        sum_uk[sw,c,:] = u_hat_plus[np.mod(iter+1, 2),c,:,k-1,sw] + sum_uk[sw,c,:] - u_hat_plus[np.mod(iter, 2),c,:,k,sw]
                        # mode spectrum
                        
                        u_hat_plus[np.mod(iter+1, 2),c,:,k,sw] = (f_hat_plus[c,:,sw] - sum_uk[sw,c,:] - lambda_hat[np.mod(iter, 2),c,:,sw]/2)/(1+self.alpha[k]*(freqs - omega_plus[np.mod(iter, 2),k, sw])**2)
                        # center frequencies
                    omega_plus[np.mod(iter+1, 2),k,sw] = np.sum(np.einsum('j,ij', freqs,(abs(u_hat_plus[np.mod(iter+1, 2), :, :, k, sw])**2)))/np.sum(abs(u_hat_plus[np.mod(iter+1, 2),:,:,k,sw])**2)
            lambda_hat[np.mod(iter+1, 2),:,:,:] = lambda_hat[np.mod(iter, 2),:,:,:] + self.tau*(np.sum(u_hat_plus[np.mod(iter+1, 2),:,:,:,:],axis = 2)-f_hat_plus)
            uDiff = np.spacing(np.ones((C, N, self.K)))
            for i in range(self.K):
                for c in range(C):
                    for n in range(N):
                        delta = u_hat_plus[0,c,:,i,n]-u_hat_plus[1,c,:,i, n]
                        delta = delta.reshape(-1)
                        uDiff[c,n,i] = (1/len(delta))*np.dot(delta,np.conj(delta)).real
            uDiff = uDiff.mean(-1)
            uDiff = uDiff.max()
            if uDiff < self.tol and iter>2:
                break
            pbar.set_description("Processing "  + "{:10.9f}".format(uDiff) + ":")
        u_hat = u_hat_plus[np.mod(iter, 2)]
        self.u_hat_cache = u_hat_plus[np.mod(iter, 2), :, :, :, -self.cache_size:]
        omega = omega_plus[np.mod(iter, 2)]
        seqw = np.argsort(omega, axis=0)
        for n in range(N):
            u_hat[:,:,:,n] = u_hat[:,:,seqw[:,n],n]
            omega[:,n] = omega[seqw[:,n],n]
        return u_hat, omega
    
    def postprocess(self, u_hat):
        u = np.zeros([self.K, u_hat.shape[0], u_hat.shape[-1]+np.sum(self.padwidth)])
        wn = window_norm(self.window, self.hop_len, self.n_fft, u_hat.shape[-1], self.win_exp)
        for k in range(self.K):
            xbuf = irfft(u_hat[:,:,k,:], n=self.n_fft, axis=1, workers=-1).real
            # xbuf = fftshift(xbuf, axes=0)
            u[k,:,:] = unbuffer(xbuf, self.window, self.hop_len, self.win_exp)
        u = u/wn.reshape(1,1,-1)

        u = u[:, :, self.padwidth[0]:-self.padwidth[1]]
        return u

In [ ]:
def validate_config(
    waveform,
    K,
    alpha,
    tau,
    tol,
    max_iters,
    vmd_n_fft,
    stvmd_window_length,
    plot_max_hz,
):
    if waveform.values.ndim != 1 or not np.isfinite(waveform.values).all():
        raise ValueError("Waveform must be a finite one-dimensional array")
    if not isinstance(K, (int, np.integer)) or K < 2:
        raise ValueError("K must be an integer not smaller than 2")
    if not isinstance(max_iters, (int, np.integer)) or max_iters < 2:
        raise ValueError("MAX_ITERS must be an integer not smaller than 2")
    if not isinstance(vmd_n_fft, (int, np.integer)) or vmd_n_fft < 2:
        raise ValueError("VMD_N_FFT must be an integer not smaller than 2")
    if (
        not isinstance(stvmd_window_length, (int, np.integer))
        or stvmd_window_length < 2
        or stvmd_window_length > waveform.values.size
    ):
        raise ValueError(
            "STVMD_WINDOW_LENGTH must be between 2 and the sample count"
        )
    for name, value in (
        ("ALPHA", alpha),
        ("TAU", tau),
        ("TOL", tol),
        ("PLOT_MAX_HZ", plot_max_hz),
    ):
        if not np.isfinite(value) or value <= 0:
            raise ValueError(f"{name} must be a finite positive number")


def estimate_vmd_memory_gb(channels, samples, K, n_fft, max_iters):
    padded = samples + n_fft - 1
    bins = padded // 2 + 1
    bytes_total = (
        max_iters * channels * bins * K * 16
        + max_iters * channels * bins * 16
        + max_iters * K * 8
    )
    return bytes_total / (1024 ** 3)


def estimate_stvmd_memory_gb(
    channels, samples, K, window_length, max_iters
):
    frames = samples
    bins = window_length // 2 + 1
    bytes_total = (
        2 * channels * bins * K * frames * 16
        + 2 * channels * bins * frames * 16
        + frames * channels * bins * 16
        + max_iters * K * frames * 8
    )
    return bytes_total / (1024 ** 3), frames


def run_original_vmd(
    x,
    fs,
    K=4,
    alpha=50.0,
    tau=1e-5,
    tol=1e-9,
    max_iters=1000,
    n_fft=64,
):
    x = np.asarray(x, dtype=float)
    model = VMD(
        num_channel=x.shape[0],
        n_fft=n_fft,
        alpha=alpha,
        K=K,
        tol=tol,
        tau=tau,
        maxiters=max_iters,
    )
    f_hat = model.prepare_offline(x)
    mode_spectrum, omega = model.apply(f_hat)
    modes = model.postprocess(mode_spectrum)
    return {
        "modes": modes,
        "mode_spectrum": mode_spectrum,
        "center_frequency_hz": omega * (fs / 2.0),
        "dynamic": False,
    }


def run_original_stvmd(
    x,
    fs,
    K=4,
    alpha=50.0,
    tau=1e-5,
    tol=1e-9,
    max_iters=1000,
    window_length=512,
):
    x = np.asarray(x, dtype=float)
    window = scipy.signal.windows.hamming(
        window_length, sym=False
    )
    model = STVMD(
        num_channel=x.shape[0],
        n_fft=window_length,
        window_func=window,
        alpha=alpha,
        K=K,
        tol=tol,
        tau=tau,
        maxiters=max_iters,
    )
    f_hat, windowed = model.prepare_offline(x)
    mode_spectrum, omega = model.apply(f_hat, dynamic=True)
    modes = model.postprocess(mode_spectrum)
    return {
        "modes": modes,
        "mode_spectrum": mode_spectrum,
        "center_frequency_hz": omega * (fs / 2.0),
        "windowed_signal": windowed,
        "dynamic": True,
        "hop_length": model.hop_len,
    }


def single_sided_amplitude(modes, fs):
    values = np.asarray(modes, dtype=float)[:, 0, :]
    sample_count = values.shape[-1]
    spectrum = np.fft.rfft(values, axis=-1)
    amplitude = np.abs(spectrum) / sample_count
    if sample_count % 2 == 0:
        amplitude[:, 1:-1] *= 2.0
    else:
        amplitude[:, 1:] *= 2.0
    frequency_hz = np.fft.rfftfreq(sample_count, d=1.0 / fs)
    return frequency_hz, amplitude


def modal_metrics(modes, fs):
    frequency_hz, amplitude = single_sided_amplitude(modes, fs)
    energy = np.sum(np.asarray(modes, dtype=float)[:, 0, :] ** 2, axis=1)
    total = float(np.sum(energy))
    energy_fraction = (
        energy / total
        if total > np.finfo(float).eps
        else np.zeros_like(energy)
    )
    return {
        "frequency_hz": frequency_hz,
        "amplitude": amplitude,
        "energy": energy,
        "energy_fraction": energy_fraction,
    }


def add_modal_metrics(raw_result, fs):
    result = dict(raw_result)
    result.update(modal_metrics(raw_result["modes"], fs))
    return result

In [ ]:
def component_label(index):
    return "Residual" if index == 0 else f"Mode {index}"


def plot_modal_time(method, time_s, result):
    modes = result["modes"][:, 0, :]
    figure, axes = plt.subplots(
        modes.shape[0],
        1,
        figsize=(11, 2.0 * modes.shape[0]),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    for index, axis in enumerate(axes):
        axis.plot(time_s, modes[index], lw=0.75)
        axis.axvline(0.0, color="black", ls="--", lw=0.6)
        axis.set_ylabel(f"{component_label(index)}\n(mm/s)")
    axes[0].set_title(f"{method}: modal time histories")
    axes[-1].set_xlabel("Time (s)")
    return figure


def plot_modal_frequency(method, fs, result, plot_max_hz):
    amplitude = result["amplitude"]
    frequency_hz = result["frequency_hz"]
    figure, axes = plt.subplots(
        amplitude.shape[0],
        1,
        figsize=(11, 2.0 * amplitude.shape[0]),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    for index, axis in enumerate(axes):
        axis.plot(frequency_hz, amplitude[index], lw=0.8)
        axis.set_ylabel(f"{component_label(index)}\n(mm/s)")
        axis.set_xlim(0.0, min(float(plot_max_hz), fs / 2.0))
    axes[0].set_title(f"{method}: modal amplitude spectra")
    axes[-1].set_xlabel("Frequency (Hz)")
    return figure


def plot_energy_fraction(method, result):
    fractions = result["energy_fraction"]
    positions = np.arange(fractions.size)
    figure, axis = plt.subplots(
        1, 1, figsize=(10, 4.5), constrained_layout=True
    )
    bars = axis.bar(positions, fractions)
    axis.set_xticks(
        positions,
        [component_label(index) for index in positions],
    )
    axis.set(
        xlabel="Component",
        ylabel="Energy fraction",
        title=f"{method}: modal energy fractions",
    )
    axis.bar_label(bars, fmt="%.3f")
    return figure


def plot_method_results(method, time_s, fs, result, plot_max_hz):
    return {
        "time_modes": plot_modal_time(method, time_s, result),
        "frequency_modes": plot_modal_frequency(
            method, fs, result, plot_max_hz
        ),
        "energy_fraction": plot_energy_fraction(method, result),
    }

In [ ]:
def save_analysis(
    output_dir,
    waveform,
    vmd_result,
    stvmd_result,
    vmd_figures,
    stvmd_figures,
    parameters,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for method, figures in (
        ("vmd", vmd_figures),
        ("stvmd", stvmd_figures),
    ):
        for name, figure in figures.items():
            figure.savefig(
                output_dir / f"{method}_{name}.png",
                dpi=300,
                bbox_inches="tight",
            )
    np.savez_compressed(
        output_dir / "vmd_stvmd_single_waveform_results.npz",
        input_file=str(waveform.path),
        direction=waveform.direction,
        fs=waveform.fs,
        time_s=waveform.time_s,
        input_velocity=waveform.values,
        vmd_modes=vmd_result["modes"],
        vmd_frequency_hz=vmd_result["frequency_hz"],
        vmd_amplitude=vmd_result["amplitude"],
        vmd_energy=vmd_result["energy"],
        vmd_energy_fraction=vmd_result["energy_fraction"],
        vmd_center_frequency_hz=vmd_result["center_frequency_hz"],
        stvmd_modes=stvmd_result["modes"],
        stvmd_frequency_hz=stvmd_result["frequency_hz"],
        stvmd_amplitude=stvmd_result["amplitude"],
        stvmd_energy=stvmd_result["energy"],
        stvmd_energy_fraction=stvmd_result["energy_fraction"],
        stvmd_center_frequency_hz=stvmd_result["center_frequency_hz"],
        parameter_names=np.asarray(list(parameters), dtype=str),
        parameter_values=np.asarray(
            [str(value) for value in parameters.values()], dtype=str
        ),
    )

## Load and validate one waveform

In [ ]:
waveform = load_single_waveform(INPUT_FILE, DIRECTION)
validate_config(
    waveform,
    K,
    ALPHA,
    TAU,
    TOL,
    MAX_ITERS,
    VMD_N_FFT,
    STVMD_WINDOW_LENGTH,
    PLOT_MAX_HZ,
)
vmd_memory_gb = estimate_vmd_memory_gb(
    1, waveform.values.size, K, VMD_N_FFT, MAX_ITERS
)
stvmd_memory_gb, stvmd_window_count = estimate_stvmd_memory_gb(
    1,
    waveform.values.size,
    K,
    STVMD_WINDOW_LENGTH,
    MAX_ITERS,
)
print("Input:", waveform.path.resolve())
print("Direction:", waveform.direction)
print("Sampling rate:", waveform.fs, "Hz")
print("Samples:", waveform.values.size)
print("STVMD hop length: 1")
print("STVMD window count:", stvmd_window_count)
print(f"Estimated VMD memory: {vmd_memory_gb:.2f} GB")
print(f"Estimated STVMD memory: {stvmd_memory_gb:.2f} GB")
if max(vmd_memory_gb, stvmd_memory_gb) > 4.0:
    warnings.warn(
        "Estimated solver memory exceeds 4 GB; reduce K, MAX_ITERS, "
        "or STVMD_WINDOW_LENGTH if necessary.",
        RuntimeWarning,
    )

## 1. Original VMD

In [ ]:
vmd_result = run_original_vmd(
    waveform.values.reshape(1, -1),
    fs=waveform.fs,
    K=K,
    alpha=ALPHA,
    tau=TAU,
    tol=TOL,
    max_iters=MAX_ITERS,
    n_fft=VMD_N_FFT,
)
vmd_result = add_modal_metrics(vmd_result, waveform.fs)
vmd_figures = plot_method_results(
    "VMD", waveform.time_s, waveform.fs, vmd_result, PLOT_MAX_HZ
)
for figure in vmd_figures.values():
    display(figure)

## 2. Original dynamic STVMD

In [ ]:
stvmd_result = run_original_stvmd(
    waveform.values.reshape(1, -1),
    fs=waveform.fs,
    K=K,
    alpha=ALPHA,
    tau=TAU,
    tol=TOL,
    max_iters=MAX_ITERS,
    window_length=STVMD_WINDOW_LENGTH,
)
stvmd_result = add_modal_metrics(stvmd_result, waveform.fs)
stvmd_figures = plot_method_results(
    "STVMD",
    waveform.time_s,
    waveform.fs,
    stvmd_result,
    PLOT_MAX_HZ,
)
for figure in stvmd_figures.values():
    display(figure)

## Save requested outputs

In [ ]:
OUTPUT_DIR = Path("output/vmd_stvmd_single_waveform")
parameters = {
    "K": K,
    "ALPHA": ALPHA,
    "TAU": TAU,
    "TOL": TOL,
    "MAX_ITERS": MAX_ITERS,
    "VMD_N_FFT": VMD_N_FFT,
    "STVMD_WINDOW_LENGTH": STVMD_WINDOW_LENGTH,
    "HOP_LENGTH": 1,
    "PLOT_MAX_HZ": PLOT_MAX_HZ,
}
if SAVE_OUTPUTS:
    save_analysis(
        OUTPUT_DIR,
        waveform,
        vmd_result,
        stvmd_result,
        vmd_figures,
        stvmd_figures,
        parameters,
    )
    print("Saved to:", OUTPUT_DIR.resolve())
else:
    print("SAVE_OUTPUTS=False: no files were written")